In [ ]:
# Basic GAN on MNIST (MLP)

#import Libraries
import os
import math
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers

In [ ]:
# 1) Config & Adding Noise
LATENT_DIM = 100
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 2e-4
BETA_1 = 0.5               # Adam beta1 for GANs for sensitivity reduction
SAMPLE_DIR = "samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)
SEED = tf.random.normal([16, LATENT_DIM])  # fixed seed for snapshots (4x4 grid)

In [ ]:
#2) Load the Dataset
# MNIST: scale to [-1, 1] for tanh generator
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32")
x_train = (x_train - 127.5) / 127.5  # [-1, 1]
x_train = np.expand_dims(x_train, axis=-1)  # (N, 28, 28, 1)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(BATCH_SIZE, drop_remainder=True)


In [ ]:
# 3) Build the Models
def build_generator(latent_dim=LATENT_DIM):
    model = tf.keras.Sequential(name="generator")
    model.add(layers.Input(shape=(latent_dim,)))
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(1024))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.BatchNormalization())
    # Output: 28*28*1 with tanh for [-1, 1]
    model.add(layers.Dense(28 * 28 * 1, activation="tanh"))
    model.add(layers.Reshape((28, 28, 1)))
    return model

def build_discriminator():
    model = tf.keras.Sequential(name="discriminator")
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Flatten())
    model.add(layers.Dense(512))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))
    model.add(layers.Dense(256))
    model.add(layers.LeakyReLU(0.2))
    model.add(layers.Dropout(0.3))
    # Output: probability real/fake
    model.add(layers.Dense(1, activation="sigmoid"))
    return model

generator = build_generator()
discriminator = build_discriminator()

In [ ]:
# 4) calculate losses and use optimizers
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)
g_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)
d_opt = tf.keras.optimizers.Adam(LEARNING_RATE, beta_1=BETA_1)


In [ ]:
# 5) Train the Models
@tf.function
def train_step(real_images):
    batch_size = tf.shape(real_images)[0]

    # Train Discriminator
    noise = tf.random.normal([batch_size, LATENT_DIM])
    fake_images = generator(noise, training=True)

    # Label smoothing + noisy labels can help stability
    real_labels = tf.ones((batch_size, 1)) * 0.9    # smooth real=0.9
    fake_labels = tf.zeros((batch_size, 1))

    with tf.GradientTape() as d_tape:
        pred_real = discriminator(real_images, training=True)
        pred_fake = discriminator(fake_images, training=True)
        d_loss_real = bce(real_labels, pred_real)
        d_loss_fake = bce(fake_labels, pred_fake)
        d_loss = d_loss_real + d_loss_fake

    d_grads = d_tape.gradient(d_loss, discriminator.trainable_variables)
    d_opt.apply_gradients(zip(d_grads, discriminator.trainable_variables))

    # Train Generator (want D(fake) -> 1)
    noise = tf.random.normal([batch_size, LATENT_DIM])
    with tf.GradientTape() as g_tape:
        fake_images = generator(noise, training=True)
        pred_fake = discriminator(fake_images, training=True)
        g_loss = bce(tf.ones((batch_size, 1)), pred_fake)

    g_grads = g_tape.gradient(g_loss, generator.trainable_variables)
    g_opt.apply_gradients(zip(g_grads, generator.trainable_variables))

    return d_loss, g_loss

In [ ]:
# 6) Save the samples
def save_samples(epoch, rows=4, cols=4):
    gen_imgs = generator(SEED, training=False).numpy()
    gen_imgs = (gen_imgs * 127.5 + 127.5).astype(np.uint8)  # back to [0,255]

    fig, axes = plt.subplots(rows, cols, figsize=(cols*2, rows*2))
    idx = 0
    for r in range(rows):
        for c in range(cols):
            axes[r, c].imshow(gen_imgs[idx, :, :, 0], cmap="gray")
            axes[r, c].axis("off")
            idx += 1
    fig.suptitle(f"Samples @ epoch {epoch}", fontsize=14)
    out_path = os.path.join(SAMPLE_DIR, f"epoch_{epoch:04d}.png")
    plt.tight_layout()
    plt.savefig(out_path)
    plt.close(fig)
    print(f"[Saved] {out_path}")

In [ ]:
# 7) Train Loop
for epoch in range(1, EPOCHS + 1):
    d_losses, g_losses = [], []
    for real_batch in train_ds:
        d_loss, g_loss = train_step(real_batch)
        d_losses.append(d_loss.numpy())
        g_losses.append(g_loss.numpy())

    print(f"Epoch {epoch}/{EPOCHS} | D loss: {np.mean(d_losses):.4f} | G loss: {np.mean(g_losses):.4f}")
    if epoch % 5 == 0 or epoch == 1:
        save_samples(epoch)

print("Training complete. Check the 'samples/' folder for generated images.")

Epoch 1/50 | D loss: 0.8114 | G loss: 2.3429
[Saved] samples/epoch_0001.png
Epoch 2/50 | D loss: 1.2764 | G loss: 0.8865
Epoch 3/50 | D loss: 1.2200 | G loss: 1.0111
Epoch 4/50 | D loss: 1.1812 | G loss: 1.1123
Epoch 5/50 | D loss: 1.1872 | G loss: 1.1194
[Saved] samples/epoch_0005.png
Epoch 6/50 | D loss: 1.2059 | G loss: 1.0899
Epoch 7/50 | D loss: 1.2302 | G loss: 1.0551
Epoch 8/50 | D loss: 1.2501 | G loss: 1.0258
Epoch 9/50 | D loss: 1.2713 | G loss: 0.9999
Epoch 10/50 | D loss: 1.2842 | G loss: 0.9779
[Saved] samples/epoch_0010.png
Epoch 11/50 | D loss: 1.2988 | G loss: 0.9582
Epoch 12/50 | D loss: 1.3047 | G loss: 0.9448
Epoch 13/50 | D loss: 1.3140 | G loss: 0.9311
Epoch 14/50 | D loss: 1.3240 | G loss: 0.9151
Epoch 15/50 | D loss: 1.3314 | G loss: 0.9042
[Saved] samples/epoch_0015.png
Epoch 16/50 | D loss: 1.3364 | G loss: 0.8940
Epoch 17/50 | D loss: 1.3367 | G loss: 0.8920
Epoch 18/50 | D loss: 1.3400 | G loss: 0.8888
Epoch 19/50 | D loss: 1.3438 | G loss: 0.8835
Epoch 20/50